# pipeverify in 5 minutes

A pipeline change can run green and keep the schema while quietly damaging the data.
This notebook shows pipeverify catching three such changes on a small orders table.
No setup needed: run the cells in order.

[GitHub](https://github.com/akshaysharma1088/pipeverify) · [PyPI](https://pypi.org/project/pipeverify/)

In [ ]:
%pip install -q pipeverify pandas

## 1. A table before the change

In [ ]:
import pandas as pd
import pipeverify

before = pd.DataFrame({
    "order_id": range(1, 1001),
    "store_id": [i % 20 for i in range(1000)],
    "region": ["North", "South", "East", "West"] * 250,
    "amount": [round(20 + (i * 7) % 80 + 0.99, 2) for i in range(1000)],
})
before.head()

## 2. A join that fans out

A new join to a store table that has two rows for store 7 duplicates that store's orders.
The job succeeds and the schema is unchanged.

In [ ]:
after = pd.concat([before, before[before.store_id == 7]], ignore_index=True)

result = pipeverify.verify(before, after, key="order_id")
print(result.verdict)
for f in result.findings:
    print(f.severity.upper(), f.message)

## 3. A unit change

Upstream starts sending West region amounts in cents. Same column, new meaning.

In [ ]:
after = before.copy()
after.loc[after.region == "West", "amount"] *= 100

result = pipeverify.verify(before, after, key="order_id")
print(result.verdict)
for f in result.findings:
    print(f.severity.upper(), f.message)

## 4. An intended change

The change was meant to restate amounts for a few refunded orders. List the column in
`expect_change` and pipeverify reports it without raising the verdict.

In [ ]:
after = before.copy()
after.loc[after.order_id.isin([5, 17, 42]), "amount"] -= 10

result = pipeverify.verify(before, after, key="order_id", expect_change=["amount"])
print(result.verdict)

## 5. Check new data against the raw source

A diff only sees rows that existed before. For new data, compare with the raw source.
Here the source sent two different amounts for some orders and the pipeline silently kept one.

In [ ]:
raw = pd.concat([before, before.head(30).assign(amount=lambda d: d.amount + 5)], ignore_index=True)
output = before  # the pipeline kept the first copy of each order

result = pipeverify.verify(None, output, key="order_id",
                           source={"data": raw, "columns": {"amount": "amount"}})
print(result.verdict)
for f in result.findings:
    print(f.severity.upper(), f.message)

## Next steps

- Use it in CI with the [GitHub Action](https://github.com/akshaysharma1088/pipeverify#github-action)
- Compare warehouse tables with a [config file](https://github.com/akshaysharma1088/pipeverify#config)
- Using it at work? [Tell us how](https://github.com/akshaysharma1088/pipeverify/issues/new?template=share_usage.yml). It helps the project.